# 토큰
LLM은 글자가 아니라 '토큰' 단위로 텍스트를 본다.
* 컨텍스트 윈도우 - 모델이 한 번에 처리할 수 있는 최대 토큰 수
```
"환불 원해요"  →  [ 환 ] [ ##불 ] [ 원 ] [ ##해요 ]
  (문장)           ↑ 이 조각 하나하나가 '토큰'
```


|원문|쪼개진 토큰|토큰 수|
|---|---|---|
|안녕하세요|`안녕` / `##하세요`|2|
|반품|`반품`|1|
|반품하고 싶어요|`반품` / `##하고` / `싶` / `##어요`|4|
|재입고|`재` / `##입고`|2|

# 실습: 한국어 문의 토큰화
실제 고객 문의(data/cs_inquiries.csv, 280건)를 토큰/ID로 바꿔 보며, 토큰화 과정을 눈으로 확인합니다. 현업에서 꼭 쓰는 토크나이저 비교·토큰 수로 비용 예측·특수토큰·패딩/잘림까지 확인합니다.

### 0.준비

In [ ]:
!pip install -q transformers pandas          # 코랩 기본 미설치 라이브러리 설치 (-q : 로그 숨김)
from google.colab import drive               # 구글 드라이브를 코랩에 연결하는 코랩 전용 도구
drive.mount('/content/drive')                # 내 드라이브를 /content/drive 경로에 마운트(연결) → 데이터 파일을 읽기 위함
from pathlib import Path                      # 경로를 / 연산자로 깔끔하게 다루는 표준 라이브러리
import pandas as pd                           # CSV 같은 표 데이터를 읽고 다루는 라이브러리
from transformers import AutoTokenizer        # 모델 이름만 주면 맞는 토크나이저를 받아주는 클래스
ROOT = Path('/content/drive/MyDrive/kt cloud tech up/gen-ai'); DATA = ROOT / 'data'  # 작업 폴더(ROOT)와 데이터 폴더(DATA) 경로 지정
tok = AutoTokenizer.from_pretrained('klue/bert-base')  # 한국어 BERT('klue/bert-base')용 토크나이저 내려받아 준비

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

### 1.토큰/ id 변환

In [ ]:
df = pd.read_csv(DATA / 'product_inquiries.csv')  # 문의 데이터 CSV를 표(DataFrame)로 읽어옴
text = df['inquiry_text'].iloc[0]           # 'inquiry_text' 열의 첫 번째 행(iloc[0]) 문장 하나를 꺼냄
print('원문:', text)                         # 토큰화하기 전 원래 문장 확인

tokens = tok.tokenize(text)                 # 원문 문자열 → 토큰(서브워드) 목록으로 쪼갬
ids = tok.convert_tokens_to_ids(tokens)     # 각 토큰 → 어휘 속 사전 번호(정수)로 변환. 모델은 이 숫자를 입력받음
print('토큰:', tokens)                       # 어떤 조각으로 나뉘었는지 관찰
print('ID  :', ids)                         # 각 토큰의 사전 번호 확인
print('토큰 수:', len(tokens))               # 몇 개 토큰으로 나뉘었는지(=모델이 처리할 단위 개수) 세어 봄

원문: 속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공간을 재보려고요.
토큰: ['속', '##건', '마이크로', '##화', '##이', '##버', '타', '##월', '3', '##매', '실제', '사이즈', '(', '가로', '##세', '##로', '##높이', ')', '가', '어떻게', '되', '##나', '##요', '?', '설치', '공간', '##을', '재보', '##려고', '##요', '.']
ID  : [1283, 2332, 25238, 2267, 2052, 2264, 1761, 2429, 23, 2077, 3966, 9471, 12, 5525, 2103, 2200, 11425, 13, 543, 3842, 859, 2075, 2182, 35, 4198, 4101, 2069, 21808, 10554, 2182, 18]
토큰 수: 31


### 3.디코딩, 특수토큰

In [ ]:
print('복원:', tok.decode(ids))             # ID 목록 → 다시 사람이 읽는 텍스트로 되돌림(디코딩). 원문과 비교

enc = tok(text)                             # 토크나이저를 함수처럼 호출 → 특수토큰·attention_mask까지 자동 포함된 인코딩(dict)
print('input_ids:', enc['input_ids'][:10])  # 모델에 들어갈 ID 목록의 앞 10개만 미리보기([:10] : 길어서 일부만)
print('토큰(특수 포함):', tok.convert_ids_to_tokens(enc['input_ids'])[:10])  # ID를 토큰으로 되돌려 봄 → 맨 앞 [CLS] 등 특수토큰 확인
print('attention_mask:', enc['attention_mask'][:10])  # 실제 토큰=1, 패딩=0. 모델에게 진짜 내용 위치를 알려주는 표식(앞 10개)

복원: 속건 마이크로화이버 타월 3매 실제 사이즈 ( 가로세로높이 ) 가 어떻게 되나요? 설치 공간을 재보려고요.
input_ids: [2, 1283, 2332, 25238, 2267, 2052, 2264, 1761, 2429, 23]
토큰(특수 포함): ['[CLS]', '속', '##건', '마이크로', '##화', '##이', '##버', '타', '##월', '3']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


1이면 의미있다는 것

# 문제 — 문의별 토큰 수
product_inquiries.csv 앞 5개 문의에 대해 (토큰 수, 원문)을 출력하세요.

In [ ]:
df = pd.read_csv(DATA / 'product_inquiries.csv')  # 문의 데이터 CSV를 표(DataFrame)로 읽어옴
text = df['inquiry_text'].iloc[0]           # 'inquiry_text' 열의 첫 번째 행(iloc[0]) 문장 하나를 꺼냄
print('원문:', text)                         # 토큰화하기 전 원래 문장 확인

tokens = tok.tokenize(text)                 # 원문 문자열 → 토큰(서브워드) 목록으로 쪼갬
ids = tok.convert_tokens_to_ids(tokens)     # 각 토큰 → 어휘 속 사전 번호(정수)로 변환. 모델은 이 숫자를 입력받음

for t in df['inquiry_text'].head(5):
    print(len(tok.tokenize(t)), '토큰 |', t)

원문: 속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공간을 재보려고요.
31 토큰 | 속건 마이크로화이버 타월 3매 실제 사이즈(가로세로높이)가 어떻게 되나요? 설치 공간을 재보려고요.
20 토큰 | 스탠다드 후드 집업 소재가 비치는 편인가요? 안에 이너 받쳐 입어야 할까요?
18 토큰 | 오래 써도 녹슬거나 휘지 않나요? 내구성이 어떤지 궁금해요.
13 토큰 | 지금 주문하면 주말 전에 받아볼 수 있을까요?
19 토큰 | 네이비 색상 실물도 화면이랑 비슷한가요? 너무 쨍한 건 아닌지 궁금해요.


# 문제 (심화) — 패딩·트렁케이션 관찰
문장 길이가 다른 두 문장을 padding=True, max_length=12, truncation=True 로 인코딩해 attention_mask 가 어떻게 달라지는지 보세요

In [ ]:
batch = [
    '반품할게요',
    '주문한 상품이 아직도 도착하지 않았는데 언제 받을 수 있나요',
    '비닐 안 뜯었는데 환불 될까요?'
]

enc = tok(
    batch,
    padding=True,
    truncation=True,
    max_length=12
)

for ids, mask in zip(enc['input_ids'], enc['attention_mask']):
    print('ids :', ids)
    print('mask:', mask)

ids : [2, 24183, 2085, 7187, 3, 0, 0, 0, 0, 0, 0, 0]
mask: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
ids : [2, 4867, 2470, 3959, 2052, 3919, 2119, 5082, 2205, 2118, 1380, 3]
mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
ids : [2, 10437, 1378, 933, 2359, 13964, 15045, 14765, 35, 3, 0, 0]
mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]


*   padding → 짧은 문장 뒤에 (0) 토큰 추가
*   truncation → 긴 문장 잘라내기
*   attention_mask -> 1 = 실제 토큰 / 0 = 패딩 토큰
*   모델은 attention_mask를 보고 PAD 부분을 무시하고 학습/추론한다.



